In [1]:
import pandas as pd

In [16]:
import pandas as pd
import keras
import os
#ok

# Create the local data directory if it doesn't exist
if not os.path.exists("data"):
    os.makedirs("data")

# Download and load the training data
f_path_1 = "data/train.csv.zip"
url_1 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/train.csv.zip"

if not os.path.exists(f_path_1):
    # Pass just the filename to get_file, it will save to your cache
    downloaded_file_1 = keras.utils.get_file("train.csv.zip", url_1)
    # Move or link it to your local path if needed, 
    # but reading directly from the source is usually safer:
    train_df = pd.read_csv(downloaded_file_1)
else:
    train_df = pd.read_csv(f_path_1)

# Download and load the test data
f_path_2 = "data/test.csv.zip"
url_2 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/test.csv.zip"

if not os.path.exists(f_path_2):
    downloaded_file_2 = keras.utils.get_file("test.csv.zip", url_2)
    test_df = pd.read_csv(downloaded_file_2)
else:
    test_df = pd.read_csv(f_path_2)

## Project 1 - NLP and Text Classification

For this project you will need to classify some angry comments into their respective category of angry. The process that you'll need to follow is (roughly):
<ol>
<li> Use NLP techniques to process the training data. 
<li> Train model(s) to predict which class(es) each comment is in.
    <ul>
    <li> A comment can belong to any number of classes, including none. 
    </ul>
<li> Generate predictions for each of the comments in the test data. 
<li> Write your test data predicitions to a CSV file, which will be scored. 
</ol>

You can use any models and NLP libraries you'd like. 

## Training Data

Use the training data to train your prediction model(s). Each of the classification output columns (toxic to the end) is a human label for the comment_text, assessing if it falls into that category of "rude". A comment may fall into any number of categories, or none at all. Membership in one output category is <b>independent</b> of membership in any of the other classes (think about this when you plan on how to make these predictions - it may also make it easier to split work amongst a team...). 

In [5]:
#train_df = pd.read_csv("train.csv.zip")
train_df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


## Test Data

In [6]:
#test_df = pd.read_csv("test.csv")
test_df.head()

,id,comment_text
0,1,Yo bitch Ja Rule is more succesful then you'll...
1,2,== From RfC == \n\n The title is fine as it is...
2,3,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,4,":If you have a look back at the source, the in..."
4,5,I don't anonymously edit articles at all.


In [7]:
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# 1. Basic Cleaning Function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\\n", " ", text) # Remove newlines
    text = re.sub(r"\W+", " ", text) # Remove special characters
    text = text.strip()
    return text

# Apply cleaning
train_df['comment_text'] = train_df['comment_text'].apply(clean_text)
test_df['comment_text'] = test_df['comment_text'].apply(clean_text)

# 2. Define labels and Features
list_classes = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
y = train_df[list_classes].values
X_text = train_df["comment_text"]

# 3. TF-IDF Vectorization
# We limit features to 20,000 to keep it performant but accurate
word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}',
    stop_words='english',
    ngram_range=(1, 2), # Using unigrams and bigrams for better context
    max_features=20000)

word_vectorizer.fit(X_text)
train_features = word_vectorizer.transform(X_text)
test_features = word_vectorizer.transform(test_df["comment_text"])

# 4. Split for internal validation
X_train, X_val, y_train, y_val = train_test_split(train_features, y, test_size=0.1, random_state=42)

print(f"Preprocessing complete. Train shape: {X_train.shape}")

Preprocessing complete. Train shape: (143613, 20000)


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GridSearchCV

# 1. Define the parameter grid for Grid Search
# We test different values of C to find the best balance between underfitting and overfitting
param_grid = {
    'estimator__C': [0.1, 1, 10, 100]
}

# 2. Set up the Grid Search with the OneVsRest wrapper
# n_jobs=-1 uses all CPU cores to speed up the search
grid_logreg = GridSearchCV(
    OneVsRestClassifier(LogisticRegression(solver='lbfgs', max_iter=500, class_weight='balanced')),
    param_grid,
    cv=3, 
    scoring='f1_weighted',
    n_jobs=-1
)

# 3. Fit the model using Grid Search
print("Starting Grid Search to find the best hyperparameters...")
grid_logreg.fit(X_train, y_train)

# 4. Set the best estimator as our final classifier
print(f"Best Parameters found: {grid_logreg.best_params_}")
classifier = grid_logreg.best_estimator_

# 5. Generate predictions using the optimized model
val_preds = classifier.predict_proba(X_val)
test_preds = classifier.predict_proba(test_features)

print("Training with Grid Search optimization complete!")

Starting Grid Search to find the best hyperparameters...
Best Parameters found: {'estimator__C': 10}
Training with Grid Search optimization complete!


In [14]:
from sklearn.metrics import accuracy_score, classification_report

# 1. Convert probabilities to binary 0/1 using a 0.5 threshold
# This is where your team can "tweak" the threshold to improve scores
threshold = 0.5
val_preds_binary = (val_preds > threshold).astype(int)

# 2. Calculate accuracy for each column independently
print("--- Validation Accuracy Scores ---")
for i, class_name in enumerate(list_classes):
    score = accuracy_score(y_val[:, i], val_preds_binary[:, i])
    print(f"{class_name}: {score:.4f}")

# 3. Overall report
print("\n--- Classification Report ---")
print(classification_report(y_val, val_preds_binary, target_names=list_classes))

--- Validation Accuracy Scores ---
toxic: 0.9387
severe_toxic: 0.9801
obscene: 0.9722
threat: 0.9940
insult: 0.9590
identity_hate: 0.9827

--- Classification Report ---
               precision    recall  f1-score   support

        toxic       0.62      0.86      0.72      1480
 severe_toxic       0.28      0.72      0.40       148
      obscene       0.69      0.87      0.77       836
       threat       0.23      0.68      0.34        37
       insult       0.56      0.84      0.67       791
identity_hate       0.31      0.72      0.43       147

    micro avg       0.57      0.84      0.68      3439
    macro avg       0.45      0.78      0.56      3439
 weighted avg       0.59      0.84      0.69      3439
  samples avg       0.06      0.08      0.07      3439



c:\Users\Shardul\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Shardul\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Shardul\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import pandas as pd
#okk
# 1. Convert test probabilities to binary labels
test_preds_binary = (test_preds > threshold).astype(int)

# 2. Create the final DataFrame
# We combine the 'id' column from the original test_df with our new predictions
submission = pd.DataFrame(test_preds_binary, columns=list_classes)
submission.insert(0, 'id', test_df['id'])

# 3. Save to CSV as requested (out.csv)
# Ensure the 'output' directory exists first
if not os.path.exists("output"):
    os.makedirs("output")

submission.to_csv('output/out.csv', index=False)
print("Submission file 'output/out.csv' has been created successfully!")

# Display the first few rows to verify format
submission.head()

Submission file 'output/out.csv' has been created successfully!


,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,1,1,1,1,0,1,1
1,2,0,0,0,0,0,0
2,3,0,0,0,0,0,0
3,4,0,0,0,0,0,0
4,5,0,0,0,0,0,0


## Output Details, Submission Info, and Example Submission

For this project, please output your predictions in a CSV file. The structure of the CSV file should match the structure of the example below. 

The output should contain one row for each row of test data, complete with the columns for ID and each classification.

Into Moodle please submit:
<ul>
<li> Your notebook file(s). I'm not going to run them, just look. 
<li> Your sample submission CSV. This will be evaluated for accuracy against the real labels; only a subset of the predictions will be scored. 
</ul>

It is REALLY, REALLY, REALLY important the the structure of your output matches the specifications. The accuracies will be calculated by a script, and it is expecting a specific format. 

### Sample Evaluator

The file prediction_evaluator.ipynb contains an example scoring function, scoreChecker. This function takes a sumbission and an answer key, loops through, and evaluates the accuracy. You can use this to verify the format of your submission. I'm going to use the same function to evaluate the accuracy of your submission, against the answer key (unless I made some mistake in this counting function).

In [14]:
#Construct dummy data for a sample output. 
#You won't do this part, you have real data
#Your data should have the same structure, so the CSV output is the same
dummy_ids = ["dfasdf234", "asdfgw43r52", "asdgtawe4", "wqtr215432"]
dummy_toxic = [0,0,0,0]
dummy_severe = [0,0,0,0]
dummy_obscene = [0,1,1,0]
dummy_threat = [0,1,0,1]
dummy_insult = [0,0,1,0]
dummy_ident = [0,1,1,0]
columns = ["id", "toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
sample_out = pd.DataFrame( list(zip(dummy_ids, dummy_toxic, dummy_severe, dummy_obscene, dummy_threat, dummy_insult, dummy_ident)),
                    columns=columns)
sample_out.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,dfasdf234,0,0,0,0,0,0
1,asdfgw43r52,0,0,1,1,0,1
2,asdgtawe4,0,0,1,0,1,1
3,wqtr215432,0,0,0,1,0,0


In [13]:
#Write DF to CSV. Please keep the "out.csv" filename. Moodle will auto-preface it with an identifier when I download it. 
#This command should work with your dataframe of predictions. 
sample_out.to_csv('output/out.csv', index=False)  

## Collaborative Project Development & Task Distribution

Our team (Shardul, Shaili, Sharry, and Jashan) has worked collectively to build this NLP pipeline. Each member contributed to a core component of the project to ensure a robust and high-performing model.

### 👥 Individual Contributions & Completed Work
* **NLP Preprocessing & Cleaning (Shaili):** Shaili led the development of the `clean_text` function to handle newlines, special characters, and case normalization.
* **Feature Engineering (Sharry):** Sharry optimized our **TF-IDF Vectorization**, managing the pipeline for 20,000 features using unigrams and bigrams.
* **Hyperparameter Tuning (Shardul):** Shardul implemented the **Grid Search** logic to find the optimal 'C' value for our **OneVsRest Logistic Regression**, ensuring the model is properly regularized.
* **Pipeline Integration & Output (Jashan):** Jashan managed the data caching and integration, ensuring all **153,164** test predictions were correctly formatted into the final `out.csv`.

### 🛠️ Final Team Steps
To finalize our submission, we have distributed the remaining responsibilities as follows:

1. **Evaluation Specialist (Shardul):** Run the final `prediction_evaluator.ipynb` using the official sample key to verify our accuracy target.
2. **Quality Lead (Shaili & Sharry):** Review the classification reports to identify if specific categories need a different probability threshold.
3. **Documentation & Formatting (Jashan):** Ensure all code cells are commented and formatted clearly to secure the **Code Quality** grade.

**Current Milestone:** The model is fully trained with Grid Search optimization complete, and the submission file is ready for final verification.

## Grading

The grading for this is split between accuracy and well written code:
<ul>
<li> 75% - Accuracy. The most accurate will get 100% on this, the others will be scaled down from there. 
<li> 25% - Code quality. Can the code be followed and made sense of - i.e. comments, sections, titles. 
</ul>